# Temporal activity trends: figures
This notebook builds figures of the temporal activity trends from the per-person JSON files saved in 004, with averages over each person's follow-up.

## Set up your environment
Set your Cloud compute profile to 32 CPUs, 208 GB RAM and 500 GB Disk, with a highmem Compute type to get that configuration.

### Expected runtime
Approximate times on the recommended VM (32 CPUs, 208 GB RAM, highmem), measured on a run of 1,991 participants.

| Run | Participants | Approx. time |
|---|---|---|
| TEST = True | 50 participants | ~40 sec |
| One chunk | ~1,000 participants | ~5.5 min |
| Full run | all participants processed by 004 (~60 chunks) | ~5.5 h |

In [ ]:
%pip install gcsfs

In [ ]:
## Import libraries
import glob
import numpy as np
import h5py
import pandas as pd
from google.cloud import storage
import json
import time
import os
import subprocess
import matplotlib.pyplot as plt
from collections import defaultdict
from datetime import datetime, timedelta
import math
from tqdm.notebook import tqdm
from joblib import Parallel, delayed
import multiprocessing

# Silence expected warnings from days or periods with no valid data (results are NaN there, which is intended)
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", message="After omitting NaNs")

NOTEBOOK_START = time.time()  # used for the total runtime printed at the end

def fmt_time(s):
    """Format seconds as h/min/s."""
    s = int(s)
    return f"{s // 3600}h {s % 3600 // 60}min {s % 60}s"


In [ ]:
os.environ["WORKSPACE_CDR"] = 'wb-silky-artichoke-2408.C2024Q3R9'

# --- PRODUCTION: workspace's own bucket (works in any copy of the workspace) ---
# Note: in this notebook bucket_ is the gs:// path and BUCKET is the local fuse mounted path (the reverse of 004)
bucket_ = subprocess.check_output(['wb', 'resource', 'resolve', '--id=multimodal_risk_prediction']).decode().strip() #gs bucket path
BUCKET  = '/home/jupyter/workspace/multimodal_risk_prediction' #local fuse mounted path
os.environ["WORKSPACE_BUCKET"] = bucket_

# Folder and file names used by this notebook (from filepaths_values.csv)
file_paths_values = pd.read_csv('filepaths_values.csv')
file_paths_values = file_paths_values[file_paths_values['notebook']=='trends_figures'][['notebook','key_','value_']]
paths_dict = file_paths_values.set_index("key_")["value_"]

# TEST = True: quick end-to-end test. 001 pulls only the first 50 EHR participants (chunk 1), and every folder/file name
# from filepaths_values.csv gets a '_test' suffix in 001-005, so a test never overwrites a full run.
# TEST is set once in 001 and read here (run 001's setup cell first).
TEST_FLAG_FILE = f'{BUCKET}/test_flag.txt'  # BUCKET is the mounted path in this notebook
assert os.path.exists(TEST_FLAG_FILE), "TEST flag not found: run the setup cell in 001 first, it saves TEST for 002-005"
TEST = open(TEST_FLAG_FILE).read().strip() == 'True'
if TEST:
    # For files the suffix goes before the extension, e.g. trends_activity_averages_test.json
    paths_dict = paths_dict.map(lambda v: '{}_test{}'.format(*os.path.splitext(str(v))))
print(f"TEST={TEST}", dict(paths_dict))

## Helper functions

In [ ]:
def list_json_files(bucket_name, folders,prefix_name):
    """
    Lists all JSON files from specified folders in a Google Cloud Storage bucket.

    Parameters:
    -----------
    bucket_name : str
        Name of the Google Cloud Storage bucket
    folders : list
        List of folder names to search within the bucket
        Example: ["jsons_calculations"]
    prefix_name : str
        Base prefix/directory name that precedes the folder structure
        Example: "trends_activity"
    Returns:
    --------
    list
        List of full blob names (paths) for all JSON files found
        Example: ["trends_activity/jsons_calculations/12345.json", ...]
    """
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    json_files = []

    for folder in folders:
        prefix = f"{prefix_name}/{folder}/"
        blobs = bucket.list_blobs(prefix=prefix)

        for blob in blobs:
            if blob.name.endswith('.json'):
                json_files.append(blob.name)

    return json_files

def read_json_file(bucket_name, file_name):
    """
    Reads and parses a JSON file from Google Cloud Storage.

    Parameters:
    -----------
    bucket_name : str
        Name of the Google Cloud Storage bucket

    file_name : str
        Full path/name of the JSON file within the bucket
        Example: "trends_activity/jsons_calculations/12345.json"

    Returns:
    --------
    dict
        Parsed JSON content

    """
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(file_name)

    content = blob.download_as_text()
    return json.loads(content)

In [ ]:
def get_ylabel(key, kk):
    """
    Y-axis label for an activity key and time period.

    Parameters:
    -----------
    key : str
        Activity type key (e.g., 'steps_limited', 'mvg_limited')

    kk : str
        Time period type ('Daily', 'Weekly', 'Monthly', 'Yearly')
        Affects whether to show 'Proportion of' (Daily) vs 'Average' (other periods)

    Returns:
    --------
    str
        Formatted y-axis label for plots
    """
    ylabel_map = {
        'steps': "Average Step count",
        'mvg': f"{'Proportion of' if kk == 'Daily' else 'Average'} moderate to vigorous minutes",
        'sd': f"{'Proportion of' if kk == 'Daily' else 'Average'} sedentary minutes",
        'lact': f"{'Proportion of' if kk == 'Daily' else 'Average'} light activity minutes"
    }

    label = next((ylabel_map[k] for k in ylabel_map if k in key))
    return label

def get_ycutoff(key, kk):
    """
    Y-axis limits for bar plots, so plots of the same activity type use a consistent scale.

    Parameters:
    -----------
    key : str
        Activity type key (e.g., 'steps_limited', 'mvg_limited')

    kk : str
        Time period type (currently unused but maintained for API consistency)

    Returns:
    --------
    tuple
        (minimum_cutoff, maximum_cutoff) for y-axis scaling
    """
    mfactor_map = {
        'steps': 1000,
        'mvg': 10,
        'sd': 100,
        'lact': 120,
    }
    afactor_map = {
        'steps': 3000,
        'mvg': 20,
        'sd': 300,
        'lact':120,
    }

    fst = next((mfactor_map[k]*10 if ('limited' in key and 'sd' in key) else mfactor_map[k] for k in mfactor_map if k in key))
    sec = next((afactor_map[k] for k in afactor_map if k in key))
    return fst,sec

def get_ycutoff_line(key, kk):
    """Y-axis limits for line plots, so plots of the same activity type use a consistent scale."""
    mfactor_map = {
        'steps': 1000,
        'hr': 10,
        'mvg': 10,
        'sd': 100,
        'lact': 120,
    }
    afactor_map = {
        'steps': 300,
        'hr': 30,
        'mvg': 2,
        'sd': 5,
        'lact':120,
    }

    fst = next((mfactor_map[k]*10 if ('limited' in key and 'sd' in key) else mfactor_map[k] for k in mfactor_map if k in key))
    sec = next((afactor_map[k] for k in afactor_map if k in key))
    return fst,sec

In [ ]:
# Heart rate follow-up per person:
# - 'person_id': participant identifier
# - 'longest_streak_start': start date of longest continuous data period
# - 'longest_streak_end': end date of longest continuous data period
# - 'longest_relaxed_streak': months of follow-up in the streak
# - 'first_active_year': year when first active

df_summary_hr = pd.read_csv(f'{bucket_}/{paths_dict.get("followup_heartrate_file")}.tsv',sep='\t')
df_summary_hr['person_id'] = df_summary_hr['person_id'].astype(int)

# Seasonal trends: people with at least 12 months of follow-up starting before 2023
# (v8 data ends in October 2023, so 2023 is not a complete year)
seasonal_hr = list(set(df_summary_hr[(df_summary_hr['first_active_year']<2023)&\
          (df_summary_hr['longest_relaxed_streak']>=12)]['person_id']))

# Yearly trends: people with at least 3 years of follow-up starting before 2023
df_3_yearly_hr = list(set(df_summary_hr[(df_summary_hr['first_active_year']<2023)&\
          (df_summary_hr['longest_relaxed_streak']>=12*3)]['person_id']))
# Yearly trends: people with at least 5 years of follow-up starting before 2023
df_5_yearly_hr = list(set(df_summary_hr[(df_summary_hr['first_active_year']<2023)&\
          (df_summary_hr['longest_relaxed_streak']>=12*5)]['person_id']))

## Person-level aggregates

Each participant's JSON from 004 already holds their average daily activity over all of their days. This step reads the JSON files in parallel and collects those averages into one table, with one row per participant, so they can be used in participant-level analyses.

In [ ]:
t0 = time.time()
# Copy the 004 JSON files to the VM so they can be read quickly
subprocess.run(f'gcloud storage cp -r {bucket_}/{paths_dict.get("trends_save")} .', shell=True, check=True)
print(f"Copy JSON files from bucket: {fmt_time(time.time() - t0)}")

In [ ]:
def read_json_local_file(name):
    """Read a local JSON file."""
    with open(name, "r") as json_file:
        return json.load(json_file)

def save_mvg_agg(file_n, keys=['steps_limited', 'mvg_limited', 'sd_limited', 'lact_limited'], metric='non_nan_avg'):
    """
    Extracts one participant's average daily activity from their JSON file.

    Parameters:
    -----------
    file_n : str
        Path to the participant's JSON file
    keys : list
        Activity metrics to extract
    metric : str
        Statistic to use (default: 'non_nan_avg')

    Returns:
    --------
    pandas.DataFrame
        One row with person_id and avg_{key} for each key
    """
    # The average over all of this person's days was already calculated in 004
    step_data = read_json_local_file(file_n)
    dict_ = {'person_id': int(file_n.split('/')[-1].split('.')[0])}
    try:
        for key in keys:
            dict_[f'avg_{key}'] = step_data["daily_one_value_act_lvl"][key][metric]
    except KeyError as e:
        print(f"KeyError: {e} in file {file_n}")
    return pd.DataFrame([dict_])

keys = ['steps_limited', 'mvg_limited', 'sd_limited', 'lact_limited']
metric = 'non_nan_avg'

# Read the files in parallel, one process per CPU
num_cores = multiprocessing.cpu_count()
st = time.time()
json_data_v = Parallel(n_jobs=num_cores)(delayed(save_mvg_agg)(file_n, keys, metric) for file_n in tqdm(glob.glob(f'{paths_dict.get("trends_save")}/*/*.json')))

df_agg = pd.concat(json_data_v)
print(f"{df_agg['person_id'].nunique()} participants")
print(f"Person-level aggregates: {fmt_time(time.time() - st)}")
df_agg.to_csv(f'{BUCKET}/{paths_dict.get("person_agg_summ_stats")}.tsv', sep='\t', index=False)

## Population-level averages

This step combines the person-level trends from 004 into population-level statistics. For each activity metric it calculates the mean, standard deviation and number of participants for daily, weekly, monthly, seasonal and yearly patterns, and saves them to one JSON file used by the plots below. Seasonal and yearly trends only include people with enough follow-up (see the cohort cell above).

In [ ]:
def save_plot_trends_arrays(bucket_name = bucket_.replace("gs://",""),\
                            folder_names = ["jsons_calculations"],  # folder written by 004
                            keys = ['mvg_limited','sd_limited','lact_limited',\
                                    'steps_limited'],\
                            metrics = ['non_nan_avg'],\
                            prefix_name = f'{paths_dict.get("trends_save")}',\
                           seasonal_hr = seasonal_hr,\
                           df_3_yearly_hr = df_3_yearly_hr,\
                           df_5_yearly_hr = df_5_yearly_hr):

    """
    Parameters:
    bucket_name: Google Cloud Storage bucket name (cleaned of 'gs://' prefix)
    folder_names: folders under prefix_name holding the per-person JSON files from 004 (default: ["jsons_calculations"])
    keys: Activity metrics to analyze ('mvg_limited', 'sd_limited', 'lact_limited', 'steps_limited')
    metrics: Statistical measures to compute (default: 'non_nan_avg')
    prefix_name: folder in the gs:// bucket to read from
    seasonal_hr: Set of participant IDs to consider for seasonal analysis
    df_3_yearly_hr: Set of participant IDs with 3+ years of data
    df_5_yearly_hr: Set of participant IDs with 5+ years of data

    Saves the results to the trends_save_avg JSON file (trends_activity_averages.json)
    """
    def nested_dict():
        return defaultdict(list)
    # Activity metrics grouped by time period; seasonal and yearly only for eligible participants
    data = defaultdict(nested_dict)
    data_seasonal = defaultdict(nested_dict)
    data_yearly_3 = defaultdict(nested_dict)
    data_yearly_5 = defaultdict(nested_dict)

    file_names = list_json_files(bucket_name, folder_names, prefix_name)
    for idx, file_n in enumerate(file_names):
        if idx % 10000 == 0:
            print(f"{idx} files processed")
        id___ = int(file_n.split('/')[-1].split('.')[0])
        json_data = read_json_file(bucket_name, file_n)
        for key in keys:
            for metric in metrics:
                # Extract data for each time period and append to the corresponding list
                try:
                    data[(key, metric)]['Daily'].append(json_data[f"daily_array_act_lvl"][key][metric])
                    data[(key, metric)]['Weekly'].append(json_data[f"weekly_stats_act_lvl"][key][metric])
                    data[(key, metric)]['Monthly'].append(json_data[f"monthly_stats_act_lvl"][key][metric])

                    if id___ in seasonal_hr:
                        data_seasonal[(key, metric)]['Seasonal'].append(json_data[f"seasonal_stats_act_lvl"][key][metric])
                    if id___ in df_3_yearly_hr:
                        data_yearly_3[(key, metric)]['Yearly_3'].append(json_data[f"yearly_stats_act_lvl_3_gap"][key][metric][0])
                    if id___ in df_5_yearly_hr:
                        data_yearly_5[(key, metric)]['Yearly_5'].append(json_data[f"yearly_stats_act_lvl_3_gap"][key][metric][0])

                except KeyError as e:
                    print(f"KeyError: {e} in file {file_n}")

    def process_data_dict(data_dict, save_data):
        """Mean, std and n per time point for daily, weekly, monthly and seasonal data."""
        for (key, metric), periods in data_dict.items():
            for period, values in periods.items():
                if not values:  # Skip empty lists
                    continue

                mean_vals = np.nanmean(values, axis=0)
                std_vals = np.nanstd(values, axis=0)
                n = np.sum(~np.isnan(values), axis=0)

                save_data[f'{period}_{key}_{metric}_mean'].append(list(mean_vals))
                save_data[f'{period}_{key}_{metric}_std'].append(list(std_vals))
                save_data[f'{period}_{key}_{metric}_n'].append([float(i) for i in list(n)])

    def process_yearly_data(data_dict, save_data):
        """Mean, std and n per follow-up year."""
        for (key, metric), periods in data_dict.items():
            for period, values in periods.items():
                if not values:
                    continue

                yearly_values = defaultdict(list)
                for entry in values:
                    for year, value in entry.items():
                        yearly_values[year].append(value)

                years = sorted(yearly_values.keys(), key=lambda x: int(x.split()[1]))
                mean_vals = [np.mean(yearly_values[year]) for year in years]
                std_vals = [np.std(yearly_values[year]) for year in years]
                n = [np.sum(~np.isnan(yearly_values[year])) for year in years]

                save_data[f'{period}_{key}_{metric}_mean'].append(list(mean_vals))
                save_data[f'{period}_{key}_{metric}_std'].append(list(std_vals))
                save_data[f'{period}_{key}_{metric}_n'].append([float(i) for i in list(n)])

    # Process all data types
    save_data = defaultdict(list)
    process_data_dict(data, save_data)
    process_data_dict(data_seasonal, save_data)
    process_yearly_data(data_yearly_3, save_data)
    process_yearly_data(data_yearly_5, save_data)

    # Save the population-level trends to a JSON file
    json_filename = f'{paths_dict.get("trends_save_avg")}'
    with open(json_filename, "w") as json_file:
        json.dump(save_data, json_file, indent=4)
    print(f"Saved {json_filename}")

st = time.time()
save_plot_trends_arrays()
et = time.time()
print(f"Population-level averages: {fmt_time(et - st)}")

## Figure 1 panels

This function plots the temporal trends of one activity metric (for example steps or moderate to vigorous minutes) across six time scales: time of day, weekday, day of month, month, and follow-up years (3 and 5 years). Each plot shows the population mean with a 95% confidence interval, using the population-level averages saved above.

Expects the JSON file of population-level averages saved above.

In [ ]:
def plot_save_trends_bar(key = 'steps_limited', metric = 'non_nan_avg', \
                         json_file_name = f'{paths_dict.get("trends_save_avg")}' ):

    """
    Parameters:
    key: activity metric to plot (default: 'steps_limited')
    metric: statistic to plot (default: 'non_nan_avg')
    json_file_name: JSON file of population-level averages saved above
    """

    timestamps = pd.date_range("00:00", "23:59", freq="1min").strftime('%H:%M')
    tick_indices = np.array(np.arange(0, 1440, 120).tolist()+[1439])
    tick_labels = [timestamps[i] for i in tick_indices]
    days_of_month = np.arange(0, 31)
    days_of_month_l = np.arange(1, 32)
    days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    month = np.arange(0, 12)
    x = np.arange(len(days))

    with open(f"{json_file_name}", "r") as json_file:
        trend_data = json.load(json_file)
    for kk,tickindices_jj,ticklabels_jj,labels_n in zip(['Daily','Weekly','Monthly','Seasonal','Yearly_3','Yearly_5'],\
                                                 [tick_indices,x,days_of_month, month,'',''],\
                                                 [tick_labels,days,days_of_month_l,months,'',''],\
                                                  ['Time of day','Weekday','Day of month','Month','Years','Years']):

        if f'{kk}_{key}_{metric}_mean' not in trend_data:
            print(f"No {kk} data (too few participants with enough follow-up), skipping")
            continue
        mean_vals = trend_data[f'{kk}_{key}_{metric}_mean'][0]
        std_vals = trend_data[f'{kk}_{key}_{metric}_std'][0]
        n = trend_data[f'{kk}_{key}_{metric}_n'][0]

        if 'Yearly' in kk:
            no_of_years = int(kk.split('_')[-1])

            years = [f'Year_{i+1}' for i in range(len(mean_vals))][0:no_of_years]
            tickindices_jj = np.arange(len(years))[0:no_of_years]
            ticklabels_jj = years[0:no_of_years]
            ci_95 = 1.96 * (std_vals[0:no_of_years]   / np.sqrt(n[0:no_of_years]  ))

        else:

            ci_95 = 1.96 * (std_vals / np.sqrt(n))

        plt.figure(figsize=(10, 5))
        if kk=='Daily':
            plt.plot(mean_vals, label="Mean", color='blue')
            plt.fill_between(range(len(mean_vals)), mean_vals - ci_95, mean_vals + ci_95, color='red', alpha=0.2, label="95% CI")
        elif 'Yearly' in kk:
            plt.bar(range(len(mean_vals))[0:no_of_years]  , mean_vals[0:no_of_years]  , color='blue', alpha=0.4, label="Mean")
            plt.errorbar(range(len(mean_vals))[0:no_of_years]  , mean_vals[0:no_of_years]  , yerr=ci_95, fmt='None', ecolor='red', capsize=5, label="95% CI")
        else:
            plt.bar(range(len(mean_vals)), mean_vals, color='blue', alpha=0.4, label="Mean")
            plt.errorbar(range(len(mean_vals)), mean_vals, yerr=ci_95, fmt='None', ecolor='red', capsize=5, label="95% CI")

        plt.xlabel(labels_n, fontsize = 15)
        if 'mvg' in key:
            plt.ylabel(get_ylabel(key, kk), fontsize = 13)
        else:
            plt.ylabel(get_ylabel(key, kk), fontsize = 15)

        plt.title(f"{kk} Trends", fontsize = 15)
        plt.legend(fontsize = 15,loc = 'upper right')
        plt.xticks(tickindices_jj,labels = ticklabels_jj, fontsize = 12)
        if '1440' in key and ('Monthly' in kk or 'Weekly' in kk):
            plt.ylim(0,int(str(np.max(mean_vals))[0])*get_ycutoff(key, kk)[0]*10+get_ycutoff(key, kk)[1])
        elif kk!='Daily' and 'Yearly' not in kk:
            plt.ylim(0,int(str(np.max(mean_vals))[0])*get_ycutoff(key, kk)[0]+get_ycutoff(key, kk)[1])
        elif 'Yearly' in kk:
            plt.ylim(0,int(str(np.max(mean_vals[0:no_of_years]))[0])*get_ycutoff(key, kk)[0]+get_ycutoff(key, kk)[1])
        plt.show()

In [ ]:
plot_save_trends_bar(key = 'steps_limited', metric = 'non_nan_avg',
                     json_file_name = f'{paths_dict.get("trends_save_avg")}')

## Socio-demographic stratification: averages

This function calculates activity trends separately for each socio-demographic group. It reads each participant's JSON file from 004, assigns the participant to their groups (age, sex, race, education, employment, income) and wear consent status, and saves the mean, standard deviation and number of participants for daily, weekly and monthly patterns.

Groups are taken from `all_person_stats.tsv` (saved in 001):

- Age: 18-44, 45-65, 66+ years
- Sex: Female, Male
- Race: White, Black, Asian, Other
- Education: advanced degree, some college degree, no college degree
- Employment: Student, Retired, Job (including self-employed), Not working (including homemaker)
- Income: >200K, 100K-200K, 50K-100K, <50K

In [ ]:
def save_plot_trends_arrays_stratified(bucket_name = bucket_.replace("gs://",""),\
                            folder_names = ["jsons_calculations"],  # folder written by 004
                            keys = ['mvg_limited','sd_limited','lact_limited',\
                                    'steps_limited'],metrics = ['non_nan_avg'],\
                                      prefix_name = f'{paths_dict.get("trends_save")}'):

    """
    Parameters:
    bucket_name: Google Cloud Storage bucket name (without 'gs://' prefix)
    folder_names: folders under prefix_name holding the per-person JSON files from 004 (default: ["jsons_calculations"])
    keys: activity metrics to analyze (e.g. 'steps_limited', 'sd_limited')
    metrics: statistics to use (default: 'non_nan_avg')
    prefix_name: folder in the gs:// bucket to read from

    Saves the results to the trends_save_avg_stratify JSON file (trends_activity_stratified_data_averages_qc.json)
    """

    df_summary = pd.read_csv(f'{bucket_}/{paths_dict.get("person_stats_file")}.tsv',sep='\t')
    wear_consent_ids = list(set(df_summary[df_summary['wear_consent']==1]['person_id']))

    groups = {
        'age': {
            '18-44': set(df_summary[(df_summary['age_at_cdr']>=18)&(df_summary['age_at_cdr']<45)]['person_id']),
            '45-65': set(df_summary[(df_summary['age_at_cdr']>=45)&(df_summary['age_at_cdr']<66)]['person_id']),
            '>65': set(df_summary[df_summary['age_at_cdr']>=66]['person_id'])
        },
        'race': {
            'white': set(df_summary[df_summary['Race']=='White']['person_id']),
            'black': set(df_summary[df_summary['Race']=='Black or African American']['person_id']),
            'asian': set(df_summary[df_summary['Race']=='Asian']['person_id']),
            'other': set(df_summary[df_summary['Race']=='Other']['person_id'])
        },
        'sex': {
            'female': set(df_summary[df_summary['sex_at_birth']=='Female']['person_id']),
            'male': set(df_summary[df_summary['sex_at_birth']=='Male']['person_id'])
        },
        'education': {
            'advanced': set(df_summary[df_summary['education_level']=='College Degree or Advanced']['person_id']),
            'some_degree': set(df_summary[df_summary['education_level']=='Some college degree']['person_id']),
            'no_degree': set(df_summary[df_summary['education_level']=='No college degree']['person_id'])
        },
        'employment': {
            'student': set(df_summary[df_summary['employment_type']=='Student']['person_id']),
            'retired': set(df_summary[df_summary['employment_type']=='Retired']['person_id']),
            'job': set(df_summary[(df_summary['employment_type']=='Self Employed')|(df_summary['employment_type']=='Job')]['person_id']),
            'not_working': set(df_summary[(df_summary['employment_type']=='Homemaker')|(df_summary['employment_type']=='Not working')]['person_id'])
        },
        'income': {
            'more_than_200k': set(df_summary[df_summary['income_level']=='>200K']['person_id']),
            '_100k_200k': set(df_summary[df_summary['income_level']=='100K-200K']['person_id']),
            '_50k_100k': set(df_summary[df_summary['income_level']=='50K-100K']['person_id']),
            'less_than_50k': set(df_summary[(df_summary['income_level']=='<10K')|(df_summary['income_level']=='10K-50K')]['person_id'])
        }
    }

    def nested_dict():
        return defaultdict(list)

    all_data = defaultdict(nested_dict)

    file_names = list_json_files(bucket_name, folder_names, prefix_name)

    for idx, file_n in enumerate(file_names):
        if idx % 10000 == 0:
            print(f"{idx} files processed")

        person_id = int(file_n.split('/')[-1].split('.')[0])
        json_data = read_json_file(bucket_name, file_n)

        is_wear_consent = person_id in wear_consent_ids

        person_groups = {}
        for type_group, group_dict in groups.items():
            for group_name, group_set in group_dict.items():
                if person_id in group_set:
                    person_groups[type_group] = group_name
                    break

        for key in keys:
            for metric in metrics:
                try:
                    daily_data = json_data[f"daily_array_act_lvl"][key][metric]
                    weekly_data = json_data[f"weekly_stats_act_lvl"][key][metric]
                    monthly_data = json_data[f"monthly_stats_act_lvl"][key][metric]

                    for type_group, group_name in person_groups.items():
                        # All participants in the group
                        all_data[(key, metric, type_group, group_name, 'all')]['Daily'].append(daily_data)
                        all_data[(key, metric, type_group, group_name, 'all')]['Weekly'].append(weekly_data)
                        all_data[(key, metric, type_group, group_name, 'all')]['Monthly'].append(monthly_data)

                        # Split by wear consent
                        consent_type = 'wear' if is_wear_consent else 'no_wear'
                        all_data[(key, metric, type_group, group_name, consent_type)]['Daily'].append(daily_data)
                        all_data[(key, metric, type_group, group_name, consent_type)]['Weekly'].append(weekly_data)
                        all_data[(key, metric, type_group, group_name, consent_type)]['Monthly'].append(monthly_data)

                except KeyError as e:
                    print(f"KeyError: {e} in file {file_n}")

    save_data = defaultdict(list)

    for (key, metric, type_group, group_name, consent_type), periods in all_data.items():
        for period, values in periods.items():
            if not values:
                continue

            values_array = np.array(values)
            mean_vals = np.nanmean(values_array, axis=0)
            std_vals = np.nanstd(values_array, axis=0)
            n = np.sum(~np.isnan(values_array), axis=0)

            prefix = f'{period}_{key}_{metric}_{type_group}_{group_name}_{consent_type}'
            save_data[f'{prefix}_mean'].append(list(mean_vals))
            save_data[f'{prefix}_std'].append(list(std_vals))
            save_data[f'{prefix}_n'].append([float(i) for i in list(n)])

    # Save the stratified trends to a JSON file
    json_filename = f'{paths_dict.get("trends_save_avg_stratify")}'
    with open(json_filename, "w") as json_file:
        json.dump(save_data, json_file)  # no indent, to keep the file small
    print(f"Saved {json_filename}")

st = time.time()
save_plot_trends_arrays_stratified()
et = time.time()
print(f"Stratified averages: {fmt_time(et - st)}")

## Socio-demographic stratification: plots

This function plots activity trends (for example step count) for each group of one socio-demographic variable, by time of day and by weekday. Plots are made for all participants and separately for participants with and without wear consent. Group sizes in the legend are counted from `all_person_stats.tsv`, using the groups listed above.

Expects the JSON file of stratified averages saved above.

In [ ]:
def plot_save_trends_stratified_bar(
    key,
    metric='non_nan_avg',
    stratify='age',
    group_names=['18-44', '45-65', '>65'],
    wear_consent_types = ['all','wear','no_wear'],
    json_file_name = f'{paths_dict.get("trends_save_avg_stratify")}'
):

    """
    Parameters:
    key: activity metric to plot (e.g. 'steps_limited', 'sd_limited')
    metric: statistic to use ('non_nan_avg' for averages ignoring missing values)
    stratify: socio-demographic variable to stratify by ('age', 'sex', 'race', 'education', 'employment', 'income')
    group_names: group labels for that variable (default: age groups)
    wear_consent_types: which participants to plot: 'all', 'wear' (wear consent) and/or 'no_wear'
    json_file_name: JSON file of stratified averages saved above
    """

    timestamps = pd.date_range("00:00", "23:59", freq="1min").strftime('%H:%M')
    tick_indices_daily = np.array(np.arange(0, 1440, 120).tolist() + [1439])
    tick_labels_daily = [timestamps[i] for i in tick_indices_daily]
    x_weekly = np.arange(7)
    labels_weekly = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

    patterns = {
        'Daily': (tick_indices_daily, tick_labels_daily, 'Time of day'),
        'Weekly': (x_weekly, labels_weekly, 'Weekday')
    }
    df_summary_i = pd.read_csv(f'{bucket_}/{paths_dict.get("person_stats_file")}.tsv',sep='\t')
    n_groups = len(group_names)
    colors_list = ['green','blue','magenta','orange','brown','cyan','purple']
    nrows, ncols = 1, 1
    for wear_consent_type in wear_consent_types:
        for kk, (tickindices_jj, ticklabels_jj, x_label) in patterns.items():
            if kk=='Weekly' and (stratify =='income' or stratify =='employment'):
                fig, axs = plt.subplots(nrows=nrows, ncols=ncols, figsize=(10, 5 * nrows), sharey=True)
            else:
                fig, axs = plt.subplots(nrows=nrows, ncols=ncols, figsize=(9 * ncols, 5 * nrows), sharey=True)

            with open(f"{json_file_name}", "r") as json_file:
                trend_data = json.load(json_file)

            for i, group_name in enumerate(group_names):
                if wear_consent_type == 'all':
                    title_add = 'all'
                    df_summary = df_summary_i[df_summary_i['wear_consent'].isin([0,1])]
                elif wear_consent_type == 'wear':
                    title_add = 'wear consent'
                    df_summary = df_summary_i[df_summary_i['wear_consent'].isin([1])]
                else:
                    title_add = 'non wear consent'
                    df_summary = df_summary_i[df_summary_i['wear_consent'].isin([0])]

                ages = [len(list(set(df_summary[(df_summary['age_at_cdr']>=18)&(df_summary['age_at_cdr']<45)]['person_id']))),\
                len(list(set(df_summary[(df_summary['age_at_cdr']>=45)&(df_summary['age_at_cdr']<66)]['person_id']))),\
                len(list(set(df_summary[(df_summary['age_at_cdr']>=66)]['person_id'])))]

                sexes = [len(list(set(df_summary[df_summary['sex_at_birth']=='Female']['person_id']))),\
                len(list(set(df_summary[df_summary['sex_at_birth']=='Male']['person_id'])))]

                races = [len(list(set(df_summary[df_summary['Race']=='White']['person_id']))),\
                len(list(set(df_summary[df_summary['Race']=='Black or African American']['person_id']))),\
                len(list(set(df_summary[df_summary['Race']=='Asian']['person_id']))),\
                len(list(set(df_summary[df_summary['Race']=='Other']['person_id'])))]

                educations = [len(list(set(df_summary[df_summary['education_level']=='College Degree or Advanced']['person_id']))),\
                len(list(set(df_summary[df_summary['education_level']=='Some college degree']['person_id']))),\
                len(list(set(df_summary[df_summary['education_level']=='No college degree']['person_id'])))]

                employments = [len(list(set(df_summary[df_summary['employment_type']=='Student']['person_id']))),\
                len(list(set(df_summary[df_summary['employment_type']=='Retired']['person_id']))),\
                len(list(set(df_summary[(df_summary['employment_type']=='Job')|\
                                       (df_summary['employment_type']=='Self Employed')]['person_id']))),\
                len(list(set(df_summary[(df_summary['employment_type']=='Not working')|\
                                       (df_summary['employment_type']=='Homemaker')]['person_id'])))]

                incomes = [len(list(set(df_summary[df_summary['income_level']=='>200K']['person_id']))),\
                len(list(set(df_summary[df_summary['income_level']=='100K-200K']['person_id']))),\
                len(list(set(df_summary[df_summary['income_level']=='50K-100K']['person_id']))),\
                len(list(set(df_summary[(df_summary['income_level']=='<10K')|\
                                       (df_summary['income_level']=='10K-50K')]['person_id'])))]

                no_dict = {'age':ages,'sex':sexes,'race':races, 'education':educations, \
                           'employment':employments,'income':incomes}
                lists_pick = no_dict[stratify]

                if f'{kk}_{key}_{metric}_{stratify}_{group_name}_{wear_consent_type}_mean' not in trend_data:
                    continue  # no participants in this group
                mean_vals = trend_data[f'{kk}_{key}_{metric}_{stratify}_{group_name}_{wear_consent_type}_mean'][0]
                std_vals = trend_data[f'{kk}_{key}_{metric}_{stratify}_{group_name}_{wear_consent_type}_std'][0]
                n = trend_data[f'{kk}_{key}_{metric}_{stratify}_{group_name}_{wear_consent_type}_n'][0]

                total_n = lists_pick[i]
                if len(mean_vals) == 0:
                    continue

                ci_95 = 1.96 * (std_vals / np.sqrt(n))
                ax = axs

                if kk == 'Daily':
                    ax.plot(mean_vals, label=f" {group_name} (n={total_n})", color=colors_list[i])
                    ax.fill_between(range(len(mean_vals)), mean_vals - ci_95, mean_vals + ci_95,
                                    color='red', alpha=0.3)
                else:
                    if kk=='Weekly' and (stratify =='income' or stratify =='employment'):
                        bar_width = 0.1
                    else:
                        bar_width = 0.2
                    x = np.arange(len(mean_vals))
                    offset = (i - n_groups / 2) * bar_width + bar_width / 2

                    ax.bar(x + offset, mean_vals, width=bar_width, color=colors_list[i], alpha=0.6, label=f" {group_name} (n={total_n})")
                    ax.errorbar(x + offset, mean_vals, yerr=ci_95, fmt='none', ecolor='black', capsize=3)

                ax.set_xlabel(x_label, fontsize = 15)
                ax.set_xticks(tickindices_jj)
                ax.set_xticklabels(ticklabels_jj, rotation=45, fontsize = 15)
                ax.set_title(f"{kk} Trends by {stratify.title()} - {title_add}", fontsize=16)
                if i == 0:
                    ax.set_ylabel(get_ylabel(key, kk), fontsize = 15)
                ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., fontsize = 15)
            plt.tight_layout(rect=[0, 0, 1, 0.95])

            plt.show()

In [ ]:
# Group labels per socio-demographic variable (must match the groups in the stratified averages above)
groups_lists = {
        'age': ['18-44','45-65','>65'],
        'race': ['white','black','asian','other'],
        'sex': ['female','male'],
        'education': ['advanced','some_degree','no_degree'],
        'employment': ['student','retired','job','not_working'],
        'income': ['more_than_200k','_100k_200k','_50k_100k','less_than_50k']
    }

In [ ]:
# Plots steps by age group. Set stratify_key to another variable (e.g. 'race') to plot that grouping.
stratify_key = 'age'

plot_save_trends_stratified_bar(key = 'steps_limited', metric = 'non_nan_avg',stratify = stratify_key,\
                                group_names = groups_lists[stratify_key],\
                                wear_consent_types = ['all','wear','no_wear'],
    json_file_name = f'{paths_dict.get("trends_save_avg_stratify")}')

In [ ]:
print(f"Total runtime of 005: {fmt_time(time.time() - NOTEBOOK_START)}")